# Exploration of datasets

In [19]:
import os, sys
from nltk.stem.snowball import SnowballStemmer
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.append(os.path.abspath(".."))
from genai_detection.detectors.impostor import ImpostorDetector
from genai_detection.visualization import datasets
from config import CONFIG
from functools import partial
import gc
import datetime
import textwrap
import pyreadstat

In [20]:
def print_min_len_text_sample(df):
    df["text_length"] = df["pair"].apply(
        lambda x: min(len(x[0].split()), len(x[1].split()))
    )
    min_len_texts = df[df["text_length"] == min(df["text_length"])]
    shorter_text = min(min_len_texts["pair"].values[0], key=lambda t: len(t.split()))
    print(
        f"Text with minimal length {len(shorter_text.split())}:\n{textwrap.fill(shorter_text, width=100)}\n"
    )


def print_max_len_text_sample(df):
    df["text_length"] = df["pair"].apply(
        lambda x: min(len(x[0].split()), len(x[1].split()))
    )
    max_len_texts = df[df["text_length"] == max(df["text_length"])]
    longer_text = max(max_len_texts["pair"].values[0], key=lambda t: len(t.split()))
    print(
        f"Text with maximal length {len(longer_text.split())}:\n{textwrap.fill(longer_text, width=100)}"
    )

# Student Essay Dataset

In [21]:
path2student_essays = (
    Path(os.getcwd()).resolve().parent / CONFIG.DATA_BASE_PATH / "student_essays/"
)
assert (
    path2student_essays.exists()
), f"Path {path2student_essays} to student essays dataset does not exist."
sav_files = list(path2student_essays.rglob("*.sav"))
assert len(sav_files) > 0, f"No .sav files found in {path2student_essays}."
for file in sav_files:
    assert file.exists(), f"File {file} does not exist."
    # print(f"\n=== {file.name} ===")
    df, meta = pyreadstat.read_sav(file, apply_value_formats=True, metadataonly=False)
    # display(df)   # Uncomment to display the dataframe in Jupyter Notebook, do not commit because sensitive data

In [22]:
import chardet

# build student essays dataset
student_essays_df = pd.DataFrame(
    columns=["author_id", "text", "task", "task_description"]
)
task_description = {
    "Ass1": "Stream of consciousness",
    "Ass2": "Talk about your childhood",
    "Ass3": "Describe your personality",
    "Ass4": "Thematic Apperception Test",
    "Ass5": "Give four examples of four different theories",
}
# iterate over all TXT files in assignment directories and extract the essays
for dir in [f"Ass{i}" for i in range(1, 6)]:
    path2ass_dir = (
        Path(os.getcwd()).resolve().parent
        / CONFIG.DATA_BASE_PATH
        / "student_essays/Intro2006"
        / dir
    )
    assert (
        path2ass_dir.exists()
    ), f'Path {path2ass_dir} to Assignment "{task_description[dir]}" of student essays dataset does not exist.'

    if dir == "Ass5":  # has subtasks (directories)
        text_files = []
        for subtask in path2ass_dir.iterdir():
            text_files.extend(subtask.glob("*.txt"))
    else:
        text_files = path2ass_dir.glob("*.txt")

    for txt_file in text_files:
        with open(txt_file, "rb") as f:
            raw_data = f.read()
            detected = chardet.detect(raw_data)
            encoding = detected["encoding"]

        essay_text = raw_data.decode(encoding)
        student_essays_df = pd.concat(
            [
                student_essays_df,
                pd.DataFrame(
                    [
                        {
                            "author_id": txt_file.stem,
                            "text": essay_text,
                            "task": txt_file.parent.name,
                            "task_description": task_description[dir],
                        }
                    ]
                ),
            ]
        )

display(student_essays_df)

# metadata dataframe
author_metadata_columns = [
    "ID",
    "TEACHER",
    "BIRTHORD",
    "SEX",
    "ETHNIC",
    "POLITOR",
    "YEAR",
]
rename_map = {
    "ID": "author_id",
    "TEACHER": "teacher",
    "BIRTHORD": "birthorder",
    "SEX": "sex",
    "ETHNIC": "ethnicity",
    "POLITOR": "political_orientation",
    "YEAR": "year",
}
raw_metadata, _ = pyreadstat.read_sav(
    Path(os.getcwd()).resolve().parent
    / CONFIG.DATA_BASE_PATH
    / "student_essays/Intro2006/2006Big5xxx.sav",
    apply_value_formats=True,
    metadataonly=False,
)
available_columns = [
    col for col in raw_metadata.columns if col in author_metadata_columns
]
author_metadata = raw_metadata[available_columns].copy()
author_metadata.rename(columns=rename_map, inplace=True)
# keep NaNs, no answer is also an answer group
display(author_metadata)

,author_id,text,task,task_description
0,2006_EFB232,I come from a small town on the coast of Texas...,Ass1,Stream of consciousness
0,2006_VBC75,"Alright, so the first thought in my head is wo...",Ass1,Stream of consciousness
0,2006_SMY244,I don't know why this kind of stuff always hap...,Ass1,Stream of consciousness
0,2006_MM22236,I feel very full at the present time. I had no...,Ass1,Stream of consciousness
0,2006_AUSTINKW,My name is Kyle Austin and my EID is austinkw....,Ass1,Stream of consciousness
...,...,...,...,...
0,JWJ296,My stepmother is extremely prone to committing...,Ass5a,Give four examples of four different theories
0,JDP2276,An example of the fundamental attribution erro...,Ass5a,Give four examples of four different theories
0,CEG588,The only thing I can think of that relates to ...,Ass5a,Give four examples of four different theories
0,KAM2437,Every once in a while I would run into a girl ...,Ass5a,Give four examples of four different theories


,author_id,teacher,sex,ethnicity,birthorder,political_orientation,year
0,144332,Gosling,female,Hispanic,oldest,middle,2.0
1,AA8763,Pennebaker,male,other,oldest,NaN,1.0
2,AAA958,Gosling,female,Hispanic,oldest,middle,1.0
3,AAH683,Pennebaker,male,Asian Amer,second,ext cons,1.0
4,AAH699,Pennebaker,female,Asian Amer,oldest,middle,1.0
...,...,...,...,...,...,...,...
1042,ZH523,Pennebaker,male,Asian Amer,oldest,som lib,3.0
1043,ZL374,Gosling,female,Hispanic,oldest,som lib,2.0
1044,ZLL62,Pennebaker,male,Asian Amer,oldest,ext lib,1.0
1045,ZRP68,Gosling,male,White,oldest,som cons,1.0


In [23]:
# display(student_essays_df[student_essays_df.task=="Ass5a"] )
# display(student_essays_df[student_essays_df.task=="Ass5b"] )
# display(student_essays_df[student_essays_df.task=="Ass5c"] )
# display(student_essays_df[student_essays_df.task=="Ass5d"] )

In [24]:
student_essays = datasets.StudentEssaysVisualization()
ds_student_essays = student_essays.load_dataset()
labels_student_essays = ds_student_essays["same"]
display(ds_student_essays.head())

,pair,authors,same
0,[i am a person who likes to be hugged. i like ...,"[SNJ244, SNJ244]",True
1,[my name is noel wat. i am a girl. i am chines...,"[NHW87, NHW87]",True
2,[it takes most people a lot to find out the re...,"[AMM2237, AMM2237]",True
3,"[loyal, confident, shy, peformer, stubborn, in...","[MNM432, MNM432]",True
4,"[a younger lady, possibly a student is putting...","[DPL253, DPL253]",True


In [25]:
student_essays_stats = student_essays.dataset_stats()

Display texts of minimal and maximal length from the student essay dataset.

The shortest Student Essay text (500 words) is a (fictional) story about theft of experiment results with a happy ending.

In [27]:
print_min_len_text_sample(ds_student_essays)

Text with minimal length 500:
there are a lot of things going on in a laboratory. not all of it deals with experiments and
research. in the picture, dr joanne and smith and seen together. dr joanne had the opportunity to do
a highly selective study on the disease effecting the bodies of small puppies. she was picked from
many scientists, including dr smith. the two women have gone to school together for their whole
lives, and the they'v also been best friends from day one. although they are friends, they're
secretely competitive with each other. therefore, when dr joanne was selected to perform the
research, dr smith was extremely sad. dr smith has been scheming a plan to ruin dr joanne's
experiment. she's been coming with dr joanne to work everyday so she can examine what she does,
where everything is, and all of joanne's lab data. her plan is to take the lab notes that she's done
with, keep them, and replace them with her made up notes that are truly obsurd. dr smith has been
pretend

The longest Student Essay text (1136 words) is a text about the inner thoughts of a person who seems to think he is manipulative and hurts women without them knowing it after a breakup.

In [28]:
print_max_len_text_sample(ds_student_essays)

Text with maximal length 1136:
as many people as there are in the world, there probably isn't any one specific thing that is unique
about me. however, the range and variety of my composition probably is unique. i think of myself as
a sort of jumbled mess that i'm trying to sort out, clean up and destroy. i -- the ego within me --
is the secretary for a mega-introvert. it's difficult for me to explain myself because i don't
really know exactly what i am. i've been sifting through the pieces for so long that i can tell you
what i possess, but i can't tell you how long i will keep any given piece, or what pieces will be
added on. i'm a shape-shifter, a chameleon, a transformer. and i'm not even referring to the
transition from high school to college. my freshman year i was silent and a lone wolf. my senior
year i was homecoming king. i've gone through almost every phase a person can go through. that's one
thing that is not unique to me (i know at least one person who is the same way -- my

In [29]:
# ds_student_essays["text_length"] = ds_student_essays["pair"].apply(lambda x: min(len(x[0].split()), len(x[1].split())))
# min_len_texts = ds_student_essays[ds_student_essays["text_length"] == min(ds_student_essays["text_length"])]
# shorter_text = min(min_len_texts['pair'].values[0], key=lambda t: len(t.split()))
# # print(f"Text with minimal length {len(shorter_text.split())}:\n{textwrap.fill(shorter_text, width=100)}\n")

# max_len_texts = ds_student_essays[ds_student_essays["text_length"] == max(ds_student_essays["text_length"])]
# longer_text = max(max_len_texts['pair'].values[0], key=lambda t: len(t.split()))
# print(f"Text with maximal length {len(longer_text.split())}:\n{textwrap.fill(longer_text, width=100)}")

In [30]:
del student_essays
del ds_student_essays
gc.collect()

465

## PAN 2023

In [31]:
# pan23 = datasets.Pan23Visualization()
# ds_pan = pan23.load_dataset()
# labels = ds_pan['same']
# display(ds_pan.head())

In [32]:
# labels.value_counts()

In [33]:
# pan23_stats = pan23.dataset_stats()

In [34]:
# pan23.plot_avg_text_length_per_author()
# pan23.plot_text_length_histogram(bins=50)
# pan23.boxplots_text_len_ngrams()

# # Python garbage collector to release memory
# del pan23
# gc.collect()

## PAN 20

In [35]:
pan20 = datasets.Pan20Visualization()
ds_pan20 = pan20.load_dataset()
labels_pan20 = ds_pan20["same"]
display(ds_pan20.head())

,id,fandoms,pair,same,authors
0,6cced668-6e51-5212-873c-717f2bc91ce6,"[Guardians of Ga'Hoole, Hetalia - Axis Powers]","[i shift a bit, warily letting my eyes dart fr...",True,"[1446633, 1446633]"
1,3c6c188a-db28-59aa-8c09-3d0f799ff579,"[Guardians of Ga'Hoole, Warriors]","[i shift a bit, warily letting my eyes dart fr...",True,"[1446633, 1446633]"
2,b0cfa94f-c9ec-5aa5-8331-a5a249b664cf,"[Guardians of Ga'Hoole, Xiaolin Showdown]",[a single tear escaped me as i left. i did hav...,True,"[1446633, 1446633]"
3,e6e86e73-9a7b-58f2-a652-a17b4a1bcabf,"[Hetalia - Axis Powers, Warriors]","[""ja."" ludwig kept his gaze upon her, solidly....",True,"[1446633, 1446633]"
4,4fe541af-912e-5a86-81a5-94c6d3891509,"[Hetalia - Axis Powers, Xiaolin Showdown]","[and he did. slowly, hesitantly...but coming f...",True,"[1446633, 1446633]"


The shortest PAN20 text (505 words) does not really make sense.

In [36]:
print_min_len_text_sample(ds_pan20)

Text with minimal length 505:
.  olympias       .        .          .               philip.      .            philip, pausanias.
pausanias            philip. ,               philip      pausanias.      olympias     euridice.
euridice   philip   .      phillip, olympias             . ,  euridice      .  olympias
euridice   .  , euridice             olympias.  , olympias  euridice                   .  olympias
.  , olympias       antipater  .         olympias   philip.          .             " .
.          .   , cassander,    ,  olympias.                .  olympias               .
",    philip.         , , ,  .  olympias                  .  olympias                      .  philip
,  olympias             .  philip      heracles,    hercules,  olympias       helen          iliad
.  iliad       .         , ,                .           . ,    , olympias          ,      :  zeus.
,           ,     ,    .       ,              .  ,           .   seer  siwa,      zeus ,  ,
.                      

The longest PAN20 text (55,406 words) seems to be a book with multiple chapters.

In [37]:
print_max_len_text_sample(ds_pan20)

Text with maximal length 55406:
chapter 1 the sun had risen on their last day in the port of sinjai, the final destination before
returning to basra with the sultan omar"s supplies. there would be a late tide so sinbad gave shore
leave to all but himself and 3 members of the crew with orders to have fun but return in shape to
weigh anchor at the evening"s high tide. firouz decided to remain on board as well. he had been
working on some experiments that needed his undivided attention. garret was in charge of the cargo.
he had to itemize the sultan"s wares. the other two crewmen were young and wet behind the ears. they
had come aboard in basra and were too excited to be a part of sinbad"s crew to care about shore
leave. they had offered to help garret in exchange for some sparring against the captain"s sword.
sinbad had learned to fight in a similar manner and in fact, was honored that they felt his skills
where worth losing a day of shore leave. unlike sinbad however, who was new at the

In [38]:
# labels_pan20.value_counts()

In [39]:
pan20_stats = pan20.dataset_stats()

In [40]:
# pan20.plot_text_length_histogram(bins=50)

In [41]:
# def normalize_text(text):
#     stemmer = SnowballStemmer('english')
#     return ' '.join(stemmer.stem(w) for w in text.lower().split())

In [42]:
# FIXME: kernel fails

# impostor_det = ImpostorDetector(ds_pan, top_n=250, portion_delete=0.5, shared_vocab_only=True)

# preproc_fn = partial(impostor_det.tokenize_char_ngrams, n=4, normalize_ws=True, space_free=True)
# pan20.plot_text_length_histogram(bins=50, preprocess_fn=preproc_fn)

In [43]:
# pan20.plot_avg_text_length_per_author()

In [44]:
# pan20.plot_avg_text_length_per_author(unit='ngrams')

In [45]:
# pan20.boxplots_text_len_ngrams()

In [46]:
# Python garbage collector to release memory
del pan20
gc.collect()

23

## Blog Corpus

In [47]:
blog = datasets.BlogVisualization()

In [48]:
ds_blog = blog.load_dataset()

The shortest blog text (500 words) is an experience report from a McLachlan concert.

In [49]:
print_min_len_text_sample(ds_blog)

Text with minimal length 500:
last night was the sarah mclachlan concert. it was so wonerful. butterfly boucher opened and was
really good. the lines were too long to purcahase her cd so i will investigate on-line. sarah sang 9
songs from afterglow and 10 from mirrorball. there were about 3 or 4 that i did not know. and she
sang black bird - i love that song. this concert is not like a madonna or any of the boy bands.
there were no dancers or gimicks. she walked out on stage in jeans and a spaghetti strap tank top
type shirt. perfect! she looks great. we had great seats. and when we felt we needed more detail
there were two large screens on either side of her. she was even wearing one of the yellow lance
armstrong bracelets. one thing that i loved was when she was on stage alone with a spot light and
sang 'you are my sunshine' as the intro to 'hold on', which was perfect. she talked to the audience
with such ease - she is good at what she does. everything was so perfect. her band left 

The longest Blog entry (8663 words) seems to be a fictional book/ story.

In [50]:
print_max_len_text_sample(ds_blog)

Text with maximal length 8663:
sink to the bottom with you by catalina chapter seventeen: all roads lead to midgar i just cant
believe this shit! cid highwind cid highwinds deep blue eyes were as thunderous and stormy as the
ocean as he stood underneath the wing of the tiny bronco, covered from head to toe in a black rain
slicker, complete with dark boots over his already booted feet. a cigarette dangled from his thin
lips, the lighted tip of it the only thing that lent illumination to the shadow cast by his plastic
hood. the light danced over his cheekbones, set high in his almost haggard face, creating hollows of
inky shadows beneath the strong bones. it also lent an orangish-yellow illumination to his deep blue
eyes, making them look like they were smoldering with fire in the literal sense as well as the
figurative. bastard, he suddenly mumbled, cigarette dancing on his thin lips as he spoke. a bit of
ash floated down to sand beneath his feet. barret turned to glare at him. you talk

In [51]:
blog_stats = blog.dataset_stats()

In [52]:
blog_df = pd.read_csv(
    Path(os.getcwd()).resolve().parent / "data/datasets/Blog_corpus/blogtext.csv"
)
blog_df["date"] = pd.to_datetime(
    blog_df["date"], format="mixed", dayfirst=True, errors="coerce"
)
print(blog_df["date"].agg(["min", "max"]))
print(
    "there is actually entries with date= 01,January,1999 and 23,August,2006, i checked it"
)

min   1999-01-01
max   2006-08-23
Name: date, dtype: datetime64[ns]
there is actually entries with date= 01,January,1999 and 23,August,2006, i checked it


In [53]:
from itertools import chain


# num_words = [len(text.split()) for text in chain.from_iterable(blog_df['pair'])]
print(blog_df.columns)
one_word_rows = blog_df[blog_df["text"].apply(lambda x: len(x.split()) == 1)]
print(f"Number of rows with one word in 'text': {len(one_word_rows)}")
print("Rows with one word in 'text':")
print(one_word_rows[["text", "date", "id"]].head(10))
print(
    "Stats are calculated on the arrow dataset, where entries must have at least 500 words, so these entries are not included in the stats."
)

Index(['id', 'gender', 'age', 'topic', 'sign', 'date', 'text'], dtype='object')
Number of rows with one word in 'text': 4276
Rows with one word in 'text':
                                                   text       date       id
1163                                             URGH   2004-08-12  1932521
1441                                     baak?          2004-08-05   589736
1543                                     niort          2004-08-05   589736
1595                                    quack.          2004-08-05   589736
1619                              TGIF!!!!!!!           2004-08-05   589736
1621                                     FLORA          2004-08-05   589736
1680                                  hmmm....          2004-08-05   589736
1719                                  marzipan          2004-08-05   589736
1737             NIORT!!!!!!!!!!!!!!!!!!!!!!!!!!!!! ... 2004-08-05   589736
1760                                       aye          2004-08-05   589736
Stats are

In [54]:
# blog.plot_avg_text_length_per_author()    # too many authors
# blog.plot_text_length_histogram(bins=50)  # FIXME: dataset comes already preprocessed
# blog.boxplots_text_len_ngrams()

In [55]:
# Python garbage collector to release memory
del blog
gc.collect()

0

## Gutenberg

In [56]:
gutenberg = datasets.GutenbergVisualization()

In [57]:
ds_gutenberg = gutenberg.load_dataset()

In [58]:
print_min_len_text_sample(ds_gutenberg)

Text with minimal length 11443:
when mr. hiram b. otis, the american minister, bought canterville chase, every one told him he was
doing a very foolish thing, as there was no doubt at all that the place was haunted. indeed, lord
canterville himself, who was a man of the most punctilious honour, had felt it his duty to mention
the f mr. otis when they came to discuss terms. "we have not cared to live in the place ourselves,"
said lord canterville, "since my grandaunt, the dowager duchess of bolton, was frightened into a
fit, from which she never really recovered, by two skeleton hands being placed on her shoulders as
she was dressing for dinner, and i feel bound to tell you, mr. otis, that the ghost has been seen by
several living members of my family, as well as by the rector of the parish, the rev. augustus
dampier, who is a fellow of king's college, cambridge. after the unfortunate accident to the
duchess, none of our younger servants would stay with us, and lady canterville often go

In [59]:
print_max_len_text_sample(ds_gutenberg)

Text with maximal length 157277:
emma woodhouse, handsome, clever, and rich, with a comfortable home and happy disposition, seemed to
unite some of the best blessings of existence; and had lived nearly twenty-one years in the world
with very little to distress or vex her. she was the youngest of the two daughters of a most
affectionate, indulgent father; and had, in consequence of her sisters marriage, been mistress of
his house from a very early period. her mother had died too long ago for her to have more than an
indistinct remembrance of her caresses; and her place had been supplied by an excellent woman as
governess, who had fallen little short of a mother in affection. sixteen years had miss taylor been
in mr. woodhouses family, less as a governess than a friend, very fond of both daughters, but
particularly of emma. between them it was more the intimacy of sisters. even before miss taylor had
ceased to hold the nominal office of governess, the mildness of her temper had hardly al

Preprocessing example on MacBeth

In [60]:
imp_det = ImpostorDetector()
# sample_text = ds_gutenberg['pair'].iloc[0][0]
sys.path.append(os.path.abspath(".."))

with open(
    Path(os.getcwd()).resolve().parent
    / Path(CONFIG.PATH2GUTENBERG).parent
    / "Macbeth_William_Shakespeare.txt",
    "r",
) as f:
    sample_text = f.read()
print(textwrap.fill(sample_text, width=80))

ACT I  SCENE I. An open Place.    Thunder and Lightning. Enter three Witches.
FIRST WITCH. When shall we three meet again? In thunder, lightning, or in rain?
SECOND WITCH. When the hurlyburly’s done, When the battle’s lost and won.  THIRD
WITCH. That will be ere the set of sun.  FIRST WITCH. Where the place?  SECOND
WITCH. Upon the heath.  THIRD WITCH. There to meet with Macbeth.  FIRST WITCH. I
come, Graymalkin!  SECOND WITCH. Paddock calls.  THIRD WITCH. Anon.  ALL. Fair
is foul, and foul is fair: Hover through the fog and filthy air.   [_Exeunt._]
SCENE II. A Camp near Forres.   Alarum within. Enter King Duncan, Malcolm,
Donalbain, Lennox, with  Attendants, meeting a bleeding Captain.  DUNCAN. What
bloody man is that? He can report, As seemeth by his plight, of the revolt The
newest state.  MALCOLM. This is the sergeant Who, like a good and hardy soldier,
fought ’Gainst my captivity.—Hail, brave friend! Say to the King the knowledge
of the broil As thou didst leave it.  SOLDIER. Dou

In [61]:
normalized_text = imp_det.preprocess_text(sample_text)
print(textwrap.fill(normalized_text, width=80))

an open place. thunder and lightning. enter three witches. when shall we three
meet again? in thunder, lightning, or in rain? when the hurlyburlys done, when
the battles lost and won. that will be ere the set of sun. where the place? upon
the heath. there to meet with macbeth. i come, graymalkin! paddock calls. anon.
fair is foul, and foul is fair: hover through the fog and filthy air. exeunt. a
camp near forres. alarum within. enter king duncan, malcolm, donalbain, lennox,
with attendants, meeting a bleeding captain. what bloody man is that? he can
report, as seemeth by his plight, of the revolt the newest state. this is the
sergeant who, like a good and hardy soldier, fought gainst my captivity.hail,
brave friend! say to the king the knowledge of the broil as thou didst leave it.
doubtful it stood; as two spent swimmers that do cling together and choke their
art. the merciless macdonwald (worthy to be a rebel, for to that the multiplying
villainies of nature do swarm upon him) from t

In [62]:
gutenberg_stats = gutenberg.dataset_stats()

In [63]:
# gutenberg.plot_avg_text_length_per_author()
# gutenberg.plot_text_length_histogram(bins=50)
# gutenberg.boxplots_text_len_ngrams()

# # Python garbage collector to release memory
# del gutenberg
# gc.collect()

## PAN 25

In [64]:
# pan25 = datasets.Pan25Visualization()
# pan25_stats = pan25.dataset_stats()

## Webis Koppel dataset

In [65]:
# koppel_webis = datasets.KoppelWebisVisualization()
# koppel_webis_stats = koppel_webis.dataset_stats()

In [66]:
# koppel_webis.plot_avg_text_length_per_author()
# koppel_webis.plot_text_length_histogram(bins=50)
# koppel_webis.boxplots_text_len_ngrams()

# # Python garbage collector to release memory
# del koppel_webis
# gc.collect()

## Compare datasets

In [67]:
combined_stats = pd.concat(
    [
        pan20_stats,
        blog_stats,
        gutenberg_stats,
        student_essays_stats,
        # pan23_stats, koppel_webis_stats, # pan25_stats
    ],
    axis=0,
    ignore_index=True,
)  # concat rows
combined_stats.set_index("dataset", inplace=True)  # set dataset as index
save_path = Path.cwd().parent / CONFIG.SAVE_PATH / "all_data" / "statistics"
save_path.mkdir(parents=True, exist_ok=True)  # create directory if it doesn't exist
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
combined_stats.to_csv(
    save_path / f"combined_stats_{timestamp}.csv", float_format="%.2f"
)
print(f"Combined statistics saved to {save_path / f'combined_stats_{timestamp}.csv'}")

Combined statistics saved to /Users/klara/Developer/Uni/MA/artificial-authorship-verification/results/all_data/statistics/combined_stats_20250718_133227.csv


In [68]:
display(combined_stats)

,num_pairs,num_authors,num_same_pairs,num_different_pairs,avg_text_len_chars,min_text_len_chars,max_text_len_chars,std_text_len_chars,avg_text_len_words,min_text_len_words,max_text_len_words,std_text_len_words,median_text_len_words
dataset,,,,,,,,,,,,,
pan20,66906,52773,35616,31290,21418.64,5364,296734,2682.43,3914.74,505,55413,512.28,3889
blog,20547,9300,10889,9658,4603.75,1152,390632,5129.62,853.32,500,70374,937.24,673
gutenberg,12,7,6,6,437870.75,63938,1696684,387405.67,78698.79,11443,297704,68329.91,60282
student_essays,3952,588,221,3731,3404.76,2399,6930,665.39,651.63,500,1362,129.22,620
